# Chapter 10 assignment: benchmarking a PINN against classical solvers

**Learning goals**

1. Run a classical work-precision sweep with `scipy.integrate.solve_ivp` and measure CPU time fairly
   (warm-up call, median of repeated calls, one thread, double precision).
2. Train a small PINN for the same problem and account for its full cost (training + evaluation).
3. Draw a work-precision diagram and read off matched-accuracy costs.
4. Compute a break-even count for a parametric surrogate and interpret it.
5. Observe the stability-limited cost of an explicit method on a stiff problem.

Everything runs on a CPU in a few minutes. The companion scripts in
`chapters/ch10_benchmarking/` implement the full benchmarks of the chapter.

## Recap

* **Accuracy**: relative $L^2$ error on a fixed uniform grid of 1001 points,
  $\varepsilon_2=\|\hat y-y\|_2/\|y\|_2$.
* **Cost**: CPU time of the single-threaded process (`time.process_time`); in addition,
  machine-independent counts (right-hand-side evaluations, residual evaluations
  $N_{\mathrm{res}} = M\times$ number of loss evaluations).
* **Matched cost** for a target $\tau$: the cost of the cheapest run with $\varepsilon_2\le\tau$.
* **Break-even count** of a surrogate: $n^\ast = C_{\mathrm{train}}/(c_{\mathrm{solve}}-c_{\mathrm{eval}})$.

In [ ]:
import os
os.environ.setdefault("OMP_NUM_THREADS", "1")      # one BLAS thread (set before NumPy is imported)
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
import time
import numpy as np
import torch
from scipy.integrate import solve_ivp

from pinnode import (ConstrainedModel, MLP, configure_torch, derivative, rel_l2, set_seed,
                     to_tensor, train, uniform)
from pinnode import plotting as P

configure_torch("float64", threads=1)
set_seed(1234)

R, Y0, T = 2.0, 0.1, 2.0
t_test = np.linspace(0.0, T, 1001)
y_exact = Y0 * np.exp(R * t_test) / (1.0 + Y0 * (np.exp(R * t_test) - 1.0))

def rhs(t, y):
    return R * y * (1.0 - y)

## Task 1: timing protocol and classical sweep

Complete `timed` (one warm-up call, then the **median** CPU time of `repeats` calls) and `sweep`
(one row per tolerance with the error on `t_test`, the CPU time and `sol.nfev`).
Use `atol = rtol` and `t_eval=t_test`.

In [ ]:
def timed(fn, repeats=5):
    # Return (result of the last call, median CPU time in seconds).
    # TODO: one warm-up call, then `repeats` timed calls with time.process_time()
    raise NotImplementedError


def sweep(method, rtols):
    rows = []
    for rtol in rtols:
        # TODO: call solve_ivp through `timed`, compute the relative L2 error with rel_l2
        raise NotImplementedError
    return rows


rtols = [10.0 ** (-3 - 0.5 * k) for k in range(19)]
results = {m: sweep(m, rtols) for m in ("RK45", "DOP853", "Radau")}
for m, rows in results.items():
    print(m, [f"{r['rel_l2']:.1e}" for r in rows[::6]])

In [ ]:
# Check your answer
assert min(r["rel_l2"] for r in results["DOP853"]) < 1e-10, "DOP853 should reach 1e-10"
assert all(r["cpu_s"] > 0 for rows in results.values() for r in rows)
print("Task 1 OK")

## Task 2: a PINN and its full cost

Train the hard-constrained PINN $y_\theta = y_0 + t N_\theta(t)$ ($16\times 2$ tanh MLP, $M=100$
uniform collocation points, 1500 Adam steps and at most 150 L-BFGS iterations). Complete the residual
in `loss_fn`. Measure the CPU time of the **whole** training plus one evaluation on `t_test`, and count
the residual evaluations $N_{\mathrm{res}}$.

In [ ]:
torch.manual_seed(1234)
model = ConstrainedModel(MLP(1, 1, width=16, depth=2, activation="tanh", domain=(0.0, T)),
                         lambda t, n: Y0 + t * n)
t_col = to_tensor(uniform(100, 0.0, T), requires_grad=True)
calls = {"n": 0}

def loss_fn():
    calls["n"] += 1
    y = model(t_col)
    r = None  # TODO: residual of y' = R y (1 - y) using derivative(y, t_col)
    L = (r ** 2).mean()
    return L, {"residual": L}

c0 = time.process_time()
hist = train(model.parameters(), loss_fn, adam_iters=1500, lr=1e-3, lr_final=1e-5,
             lbfgs_iters=150, print_every=500)
with torch.no_grad():
    y_pinn = model(to_tensor(t_test)).numpy().ravel()
pinn_cpu = time.process_time() - c0
pinn_err = rel_l2(y_pinn, y_exact)
n_res = 100 * calls["n"]
print(f"PINN: rel L2 = {pinn_err:.2e}, CPU time = {pinn_cpu:.1f} s, residual evaluations = {n_res:.2e}")

In [ ]:
# Check your answer
assert pinn_err < 1e-2, "the PINN should reach at least 1e-2 (typically 1e-4 or better)"
assert n_res >= 150000
print("Task 2 OK")

## Task 3: work-precision diagram and matched cost

Plot the classical curves (error on the horizontal axis, CPU time on the vertical axis, both
logarithmic) and the PINN point. Then complete `matched_cost` and print, for $\tau=10^{-4}$, the cost of
every method and the ratio PINN / fastest classical method.

In [ ]:
def matched_cost(rows, tau):
    # TODO: CPU time of the cheapest row with rel_l2 <= tau, or None
    raise NotImplementedError

fig, ax = P.figure(1, 1, width="half", aspect=0.9)
for k, (m, rows) in enumerate(results.items()):
    ax.loglog([r["rel_l2"] for r in rows], [r["cpu_s"] for r in rows], marker=P.MARKERS[k],
              color=P.SERIES[k], lw=0.9, ms=3, mfc="white", label=m)
ax.loglog([pinn_err], [pinn_cpu], "*", ms=9, color=P.COLORS["red"], mec="k", mew=0.5, label="PINN")
ax.set_xlabel("relative $L^2$ error"); ax.set_ylabel("CPU time [s]"); ax.legend(fontsize=6)

tau = 1e-4
costs = {m: matched_cost(rows, tau) for m, rows in results.items()}
costs["PINN"] = pinn_cpu if pinn_err <= tau else None
print(costs)

In [ ]:
# Check your answer
assert costs["DOP853"] is not None and costs["DOP853"] < 1.0
fastest = min(c for m, c in costs.items() if m != "PINN" and c is not None)
if costs["PINN"] is not None:
    print(f"PINN / fastest classical at 1e-4: {costs['PINN'] / fastest:.1e}")
print("Task 3 OK")

## Task 4: break-even count

Suppose a parametric PINN for $r\in[1,3]$ has been trained in $C_{\mathrm{train}}$ seconds, one network
evaluation costs $c_{\mathrm{eval}}$ and one classical solve $c_{\mathrm{solve}}$. Complete `break_even`
(return `float('inf')` if the network is not cheaper than a solve). Measure $c_{\mathrm{solve}}$ for RK45
at `rtol=1e-6` (trajectory on `t_test` and point value $y(2)$) and $c_{\mathrm{eval}}$ for your network
of Task 2 (1001 points and 1 point); use your training time as a stand-in for $C_{\mathrm{train}}$.

In [ ]:
def break_even(C_train, c_solve, c_eval):
    # TODO
    raise NotImplementedError

_, c_solve_traj = timed(lambda: solve_ivp(rhs, (0, T), [Y0], method="RK45", t_eval=t_test, rtol=1e-6, atol=1e-6))
_, c_solve_pt = timed(lambda: solve_ivp(rhs, (0, T), [Y0], method="RK45", rtol=1e-6, atol=1e-6))
x_traj, x_pt = to_tensor(t_test), to_tensor([T])
with torch.no_grad():
    _, c_eval_traj = timed(lambda: model(x_traj))
    _, c_eval_pt = timed(lambda: model(x_pt))
print("trajectory:", break_even(pinn_cpu, c_solve_traj, c_eval_traj))
print("point     :", break_even(pinn_cpu, c_solve_pt, c_eval_pt))

In [ ]:
# Check your answer
assert break_even(10.0, 2e-3, 1e-3) == 10.0 / 1e-3
assert break_even(10.0, 1e-3, 2e-3) == float("inf")
print("Task 4 OK")

## Task 5: stiffness and explicit methods

For the system $y'=Ay$ of the chapter with eigenvalues $-1$ and $-\lambda$, $y(0)=(2,0)$, $t\in[0,1]$,
record `sol.nfev` of RK45 and Radau (`rtol=1e-6`, analytic Jacobian for Radau) for
$\lambda\in\{10,100,1000,10000\}$. Explain the growth of the RK45 count (Exercise "Stability-limited
cost of an explicit method").

In [ ]:
def stiff_counts(lam):
    A = 0.5 * np.array([[-(lam + 1), lam - 1], [lam - 1, -(lam + 1)]])
    # TODO: return (nfev of RK45, nfev of Radau) for rtol = atol = 1e-6
    raise NotImplementedError

for lam in (10, 100, 1000, 10000):
    print(lam, stiff_counts(lam))

In [ ]:
# Check your answer
n_rk, n_radau = stiff_counts(10000)
assert n_rk > 10 * n_radau, "RK45 should need far more evaluations than Radau for lambda = 1e4"
print("Task 5 OK")

## Questions for discussion

1. Which costs did you include for the PINN, and which did you leave out (hyperparameter search)?
2. Would your conclusion change if the PINN were trained on a GPU? What would a fair comparison
   require then?
3. In which of the settings of Section "Where PINNs can pay off" would you consider a PINN for this
   problem?